# Experiment 2 — Attention + Decoding Strategies



In [ ]:
!pip -q install sacrebleu

In [ ]:
import os, re, random, math, time, copy, json, collections, unicodedata
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
import matplotlib.pyplot as plt
import sacrebleu

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. Data
Defaults: 40k training pairs, sentences up to 25 tokens, 8k vocab, batch 256.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
TRAIN_PATH = "/content/drive/MyDrive/en_fr_train.csv"   # three separate files from the shared Drive folder
VAL_PATH   = "/content/drive/MyDrive/en_fr_val.csv"
TEST_PATH  = "/content/drive/MyDrive/en_fr_test.csv"
SRC_COL, TGT_COL = 0, 1       # column index (int) or column name (str), e.g. "english_sentence", "hindi_sentence"
MAX_TRAIN_PAIRS = 40_000     # cap on training pairs for faster experiments (val/test are used in full)
MAX_LEN   = 25                # max tokens per sentence (longer pairs are dropped from all three splits)
MIN_FREQ  = 2                 # words rarer than this become <unk>
MAX_VOCAB = 8_000

def load_parallel(path, src_col=0, tgt_col=1):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        df = pd.read_csv(path)
    elif ext in (".xlsx", ".xls"):
        df = pd.read_excel(path)
    else:  # .tsv / .txt (src<TAB>tgt[<TAB>attribution])
        df = pd.read_csv(path, sep="\t", header=None, quoting=3, on_bad_lines="skip", engine="python")
    col = lambda c: df.iloc[:, c] if isinstance(c, int) else df[c]
    pairs = []
    for s, t in zip(col(src_col), col(tgt_col)):
        if isinstance(s, str) and isinstance(t, str) and s.strip() and t.strip():
            pairs.append((s.strip(), t.strip()))
    return pairs

if all(os.path.exists(p) for p in (TRAIN_PATH, VAL_PATH, TEST_PATH)):
    train_pairs = load_parallel(TRAIN_PATH, SRC_COL, TGT_COL)
    random.Random(SEED).shuffle(train_pairs); train_pairs = train_pairs[:MAX_TRAIN_PAIRS]
    val_pairs   = load_parallel(VAL_PATH,  SRC_COL, TGT_COL)
    test_pairs  = load_parallel(TEST_PATH, SRC_COL, TGT_COL)
else:
    print("!! CSV paths not found -> using a tiny TOY corpus (split 80/10/10) just to smoke-test the pipeline.")
    names = ["ram","sita","john","mary","amit","priya"]; things = ["apples","books","pens","cars","dogs"]
    hin = {"apples":"seb","books":"kitabein","pens":"kalam","cars":"gaadiyan","dogs":"kutte"}
    toy = [(f"{n} has {k} {t}", f"{n} ke paas {k} {hin[t]} hain") for n in names for t in things for k in range(1, 30)]
    random.Random(SEED).shuffle(toy); a, b = int(0.8 * len(toy)), int(0.9 * len(toy))
    train_pairs, val_pairs, test_pairs = toy[:a], toy[a:b], toy[b:]
print(f"raw pairs -> train {len(train_pairs)} | val {len(val_pairs)} | test {len(test_pairs)}"); print(train_pairs[:3])

## 2. Tokenisation and vocabularies (built from the train file only)

In [ ]:
# Unicode-aware tokenizer: keeps Devanagari words intact (Python's \w drops combining vowel signs) and
# splits off punctuation, incl. the danda "।".
TOKEN_RE = re.compile(r"[\u0964\u0965]|[^\s\w\u0900-\u097F]|[\w\u0900-\u0963\u0966-\u097F]+")
def tokenize(s): return TOKEN_RE.findall(unicodedata.normalize("NFC", s.lower()))

PAD, SOS, EOS, UNK = 0, 1, 2, 3
class Vocab:
    def __init__(self, token_lists, min_freq=2, max_size=20000):
        cnt = collections.Counter(t for toks in token_lists for t in toks)
        self.itos = ["<pad>", "<sos>", "<eos>", "<unk>"] + [w for w, c in cnt.most_common(max_size) if c >= min_freq]
        self.stoi = {w: i for i, w in enumerate(self.itos)}
    def encode(self, toks, add_sos_eos=False):
        ids = [self.stoi.get(t, UNK) for t in toks]
        return [SOS] + ids + [EOS] if add_sos_eos else ids
    def decode(self, ids):
        out = []
        for i in ids:
            if i == EOS: break
            if i in (PAD, SOS): continue
            out.append(self.itos[i])
        return out
    def __len__(self): return len(self.itos)

def prep(pairs):
    out = []
    for s, t in pairs:
        st, tt = tokenize(s), tokenize(t)
        if 1 <= len(st) <= MAX_LEN and 1 <= len(tt) <= MAX_LEN: out.append((st, tt))
    return out

# each split comes from its own file; vocabularies are built from the TRAIN file only
train, val, test = prep(train_pairs), prep(val_pairs), prep(test_pairs)
src_vocab = Vocab([s for s, _ in train], MIN_FREQ, MAX_VOCAB)
tgt_vocab = Vocab([t for _, t in train], MIN_FREQ, MAX_VOCAB)
print(f"train {len(train)} | val {len(val)} | test {len(test)} | src vocab {len(src_vocab)} | tgt vocab {len(tgt_vocab)}")

def make_items(split):
    return [dict(src_tok=s, tgt_tok=t, src_ids=src_vocab.encode(s), tgt_ids=tgt_vocab.encode(t, True)) for s, t in split]
train_items, val_items, test_items = make_items(train), make_items(val), make_items(test)

def collate(batch):
    src = pad_sequence([torch.tensor(b["src_ids"]) for b in batch], batch_first=True, padding_value=PAD)
    tgt = pad_sequence([torch.tensor(b["tgt_ids"]) for b in batch], batch_first=True, padding_value=PAD)
    lens = torch.tensor([len(b["src_ids"]) for b in batch])   # stays on CPU (needed by pack_padded_sequence)
    return src, lens, tgt

BATCH = 256
train_dl = DataLoader(train_items, batch_size=BATCH, shuffle=True,  collate_fn=collate)
val_dl   = DataLoader(val_items,   batch_size=BATCH, shuffle=False, collate_fn=collate)
test_dl  = DataLoader(test_items,  batch_size=BATCH, shuffle=False, collate_fn=collate)

## 3. Model with pluggable attention

In [ ]:
class EncoderBi(nn.Module):
    def __init__(self, V, E, H, drop):
        super().__init__()
        self.emb = nn.Embedding(V, E, padding_idx=PAD)
        self.rnn = nn.LSTM(E, H, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(drop)
        self.bridge_h, self.bridge_c = nn.Linear(2 * H, H), nn.Linear(2 * H, H)
    def forward(self, src, lens):
        x = self.drop(self.emb(src))
        packed = pack_padded_sequence(x, lens, batch_first=True, enforce_sorted=False)
        out, (h, c) = self.rnn(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=src.size(1))   # [B,S,2H] all encoder states
        h = torch.tanh(self.bridge_h(torch.cat([h[0], h[1]], -1)))                      # initial decoder state
        c = torch.tanh(self.bridge_c(torch.cat([c[0], c[1]], -1)))
        return out, h, c

class Attention(nn.Module):
    def __init__(self, kind, H, K):            # H decoder dim, K encoder-state dim (=2H)
        super().__init__()
        self.kind = kind
        if kind == "bahdanau":
            self.Wq, self.Wk, self.v = nn.Linear(H, H, bias=False), nn.Linear(K, H, bias=False), nn.Linear(H, 1, bias=False)
        else:                                   # luong "general": score = q^T W k
            self.W = nn.Linear(K, H, bias=False)
    def forward(self, q, keys, mask):           # q [B,H], keys [B,S,K], mask [B,S] (True = real token)
        if self.kind == "bahdanau":
            e = self.v(torch.tanh(self.Wq(q).unsqueeze(1) + self.Wk(keys))).squeeze(-1)
        else:
            e = torch.bmm(self.W(keys), q.unsqueeze(-1)).squeeze(-1)
        a = F.softmax(e.masked_fill(~mask, torch.finfo(e.dtype).min), dim=-1)           # alignment weights [B,S]
        ctx = torch.bmm(a.unsqueeze(1), keys).squeeze(1)            # weighted sum of encoder states [B,K]
        return ctx, a

class DecoderAttn(nn.Module):
    def __init__(self, V, E, H, K, kind, drop):
        super().__init__()
        self.kind = kind
        self.emb, self.drop = nn.Embedding(V, E, padding_idx=PAD), nn.Dropout(drop)
        self.cell = nn.LSTMCell(E + (K if kind == "bahdanau" else 0), H)
        self.attn = Attention(kind, H, K) if kind in ("bahdanau", "luong") else None
        if kind == "luong": self.comb = nn.Linear(H + K, H)
        self.out = nn.Linear(H + (K if kind == "bahdanau" else 0), V)
    def step(self, tok, h, c, enc_out, mask):
        e, a = self.drop(self.emb(tok)), None
        if self.kind == "bahdanau":
            ctx, a = self.attn(h, enc_out, mask)
            h, c = self.cell(torch.cat([e, ctx], -1), (h, c)); feat = torch.cat([h, ctx], -1)
        elif self.kind == "luong":
            h, c = self.cell(e, (h, c))
            ctx, a = self.attn(h, enc_out, mask); feat = torch.tanh(self.comb(torch.cat([ctx, h], -1)))
        else:
            h, c = self.cell(e, (h, c)); feat = h
        return self.out(self.drop(feat)), h, c, a

class NMT(nn.Module):
    def __init__(self, src_V, tgt_V, kind="bahdanau", E=128, H=256, drop=0.3):
        super().__init__()
        self.kind = kind
        self.enc = EncoderBi(src_V, E, H, drop)
        self.dec = DecoderAttn(tgt_V, E, H, 2 * H, kind, drop)
    def encode(self, src, lens):
        out, h, c = self.enc(src, lens); return out, h, c, src != PAD
    def forward(self, src, lens, tgt, tf_ratio=1.0):
        out, h, c, mask = self.encode(src, lens)
        tok, logits = tgt[:, 0], []
        for t in range(1, tgt.size(1)):
            lg, h, c, _ = self.dec.step(tok, h, c, out, mask); logits.append(lg)
            tok = tgt[:, t] if random.random() < tf_ratio else lg.argmax(-1)
        return torch.stack(logits, 1)

## 3. Training — light, resumable, time-boxed


In [ ]:
TRAIN_KINDS     = ["none", "bahdanau", "luong"]
EPOCHS          = 8
PATIENCE        = 2
TIME_BUDGET_MIN = 35
CKPT_DIR        = "/content/drive/MyDrive/seq2seq/ckpt_exp2"
USE_AMP         = device.type == "cuda"

os.makedirs(CKPT_DIR, exist_ok=True)

def run_epoch(model, loader, opt=None, scaler=None, tf=1.0):
    train = opt is not None
    model.train(train)
    total, n = 0.0, 0
    for src, lens, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        with torch.set_grad_enabled(train), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model(src, lens, tgt, tf if train else 1.0)
            loss = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1), ignore_index=PAD)
        if train:
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
        total += loss.item(); n += 1
    return total / n

def fit_resumable(model, kind, lr=1e-3, tf=0.8):
    last_p, best_p = f"{CKPT_DIR}/{kind}_last.pt", f"{CKPT_DIR}/{kind}_best.pt"
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    st = dict(epoch=0, hist=[], best=1e9, bad=0, done=False)
    if os.path.exists(last_p):
        ck = torch.load(last_p, map_location=device)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); st = ck["st"]
        print(f"[{kind}] found checkpoint at epoch {st['epoch']} (done={st['done']})")
    t_start = time.time()
    while not st["done"]:
        t0 = time.time()
        tr, va = run_epoch(model, train_dl, opt, scaler, tf), run_epoch(model, val_dl)
        st["epoch"] += 1; st["hist"].append((tr, va))
        print(f"[{kind}] epoch {st['epoch']:2d} | train {tr:.3f} | val {va:.3f} | {time.time()-t0:.0f}s")
        if va < st["best"]:
            st["best"], st["bad"] = va, 0; torch.save(model.state_dict(), best_p)
        else:
            st["bad"] += 1
            if st["bad"] >= PATIENCE: st["done"] = True; print("early stop")
        if st["epoch"] >= EPOCHS: st["done"] = True
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "st": st}, last_p)
        if not st["done"] and (time.time() - t_start) / 60 > TIME_BUDGET_MIN:
            print(f"[{kind}] time budget reached - checkpoint saved. RE-RUN THIS CELL to continue."); break
    model.load_state_dict(torch.load(best_p, map_location=device))
    return st["hist"], st["done"]

models, histories, finished = {}, {}, {}
for kind in TRAIN_KINDS:
    set_seed(); models[kind] = NMT(len(src_vocab), len(tgt_vocab), kind).to(device)
    histories[kind], finished[kind] = fit_resumable(models[kind], kind)
    if not finished[kind]: break                      # stop here; re-run the cell to resume
print({k: ("finished" if v else "PARTIAL - re-run this cell") for k, v in finished.items()})

for kind, h in histories.items(): plt.plot([x[1] for x in h], label=f"{kind} (val)")
plt.legend(); plt.xlabel("epoch"); plt.ylabel("val cross-entropy"); plt.title("Validation loss: attention vs. none"); plt.show()

## 4. Decoding strategies
One function, several strategies: **greedy**, **beam search** (length-normalised), and **sampling** with temperature / top-k / top-p (nucleus).

In [ ]:
def corpus_bleu(hyps, refs):
    # we tokenised ourselves, so tell sacreBLEU not to re-tokenise
    return sacrebleu.corpus_bleu(hyps, [refs], tokenize="none").score

def sample_from(logits, temperature=1.0, top_k=0, top_p=1.0):
    logits = logits / max(temperature, 1e-6)
    if top_k > 0:
        kth = torch.topk(logits, min(top_k, logits.numel())).values[-1]
        logits = logits.masked_fill(logits < kth, -1e9)
    if top_p < 1.0:
        sl, si = torch.sort(logits, descending=True)
        cum = F.softmax(sl, -1).cumsum(-1)
        remove = cum > top_p
        remove[1:] = remove[:-1].clone(); remove[0] = False          # always keep the top token
        drop_mask = torch.zeros_like(remove); drop_mask[si] = remove
        logits = logits.masked_fill(drop_mask, -1e9)
    return torch.multinomial(F.softmax(logits, -1), 1)

def beam_search(model, out, h, c, mask, beam=4, max_len=60, alpha=0.6):
    beams, finished = [([SOS], 0.0, h, c)], []
    for _ in range(max_len):
        cands = []
        for toks, lp, h_, c_ in beams:
            lg, h2, c2, _ = model.dec.step(torch.tensor([toks[-1]], device=device), h_, c_, out, mask)
            logp = F.log_softmax(lg, -1)[0]; top = torch.topk(logp, beam)
            for v, i in zip(top.values.tolist(), top.indices.tolist()): cands.append((toks + [i], lp + v, h2, c2))
        cands.sort(key=lambda x: x[1], reverse=True)
        beams = []
        for cand in cands:
            (finished if cand[0][-1] == EOS else beams).append(cand)
            if len(beams) == beam: break
        if len(finished) >= beam or not beams: break
    if not finished: finished = beams
    lpen = lambda n: ((5 + n) / 6) ** alpha                              # GNMT length penalty
    best = max(finished, key=lambda x: x[1] / lpen(len(x[0]) - 1))
    return best[0][1:], None

@torch.no_grad()
def translate(model, ids, strategy="greedy", beam=4, temperature=1.0, top_k=0, top_p=1.0, max_len=60, want_attn=False):
    model.eval()
    src, lens = torch.tensor([ids], device=device), torch.tensor([len(ids)])
    out, h, c, mask = model.encode(src, lens)
    if strategy == "beam": return beam_search(model, out, h, c, mask, beam, max_len)
    tok, res, atts = torch.tensor([SOS], device=device), [], []
    for _ in range(max_len):
        lg, h, c, a = model.dec.step(tok, h, c, out, mask)
        tok = lg.argmax(-1) if strategy == "greedy" else sample_from(lg[0], temperature, top_k, top_p)
        if a is not None: atts.append(a[0].cpu().numpy())
        if tok.item() == EOS: break
        res.append(tok.item())
    return res, (np.stack(atts) if (want_attn and atts) else None)

def decode_items(model, items, n=None, **kw):
    items = items[:n] if n else items
    return [" ".join(tgt_vocab.decode(translate(model, it["src_ids"], **kw)[0])) for it in items]

N_EVAL = min(300, len(test_items))          # decoding sentence-by-sentence is slow for beam search; raise if you have time
refs = [" ".join(it["tgt_tok"]) for it in test_items[:N_EVAL]]

### 4a. Attention vs. no attention (greedy)

In [ ]:
res_attn = {}
for kind, m in models.items():
    set_seed(); hyps = decode_items(m, test_items, N_EVAL, strategy="greedy")
    res_attn[kind] = (corpus_bleu(hyps, refs), hyps)
bleu_table = pd.DataFrame({"model": list(res_attn), "BLEU (greedy)": [round(v[0], 2) for v in res_attn.values()]})
print(bleu_table.to_string(index=False))

### 4b. BLEU by source length — does attention fix long sentences?

In [ ]:
src_len = np.array([len(it["src_tok"]) for it in test_items[:N_EVAL]])
edges = [(1, 5), (6, 10), (11, 15), (16, 20), (21, MAX_LEN)]
rows = []
for a, b in edges:
    mk = (src_len >= a) & (src_len <= b)
    row = {"len": f"{a}-{b}", "n": int(mk.sum())}
    for kind, (_, hyps) in res_attn.items():
        row[kind] = corpus_bleu([h for h, k in zip(hyps, mk) if k], [r for r, k in zip(refs, mk) if k]) if mk.sum() >= 10 else np.nan
    rows.append(row)
len_df = pd.DataFrame(rows); print(len_df.round(2).to_string(index=False))
len_df.plot(x="len", y=list(res_attn), kind="bar", title="BLEU vs. source length"); plt.ylabel("BLEU"); plt.show()

### 4c. Decoding-strategy comparison (on the best attention model)

In [ ]:
assert any(k != "none" for k in res_attn), "train at least one attention model first"
best_kind = max([k for k in res_attn if k != "none"], key=lambda k: res_attn[k][0])
print("best attention variant:", best_kind)
configs = [
    ("greedy",                      dict(strategy="greedy")),
    ("beam k=3",                    dict(strategy="beam", beam=3)),
    ("beam k=5",                    dict(strategy="beam", beam=5)),
    ("sample T=0.7",                dict(strategy="sample", temperature=0.7)),
    ("sample T=1.0",                dict(strategy="sample", temperature=1.0)),
    ("top-k=10 (T=0.8)",            dict(strategy="sample", temperature=0.8, top_k=10)),
    ("top-p=0.9 (nucleus)",         dict(strategy="sample", temperature=1.0, top_p=0.9)),
]
dec_rows, dec_out = [], {}
for name, kw in configs:
    stochastic = kw["strategy"] == "sample"
    scores, t0 = [], time.time()
    for run in range(2 if stochastic else 1):       # sampling is random -> average 2 runs
        set_seed(SEED + run); hyps = decode_items(models[best_kind], test_items, N_EVAL, **kw)
        scores.append(corpus_bleu(hyps, refs))
    dec_out[name] = hyps
    dec_rows.append({"strategy": name, "BLEU": round(float(np.mean(scores)), 2),
                     "std": round(float(np.std(scores)), 2) if stochastic else 0.0,
                     "sec/sentence": round((time.time() - t0) / ((2 if stochastic else 1) * N_EVAL), 4)})
dec_df = pd.DataFrame(dec_rows); print(dec_df.to_string(index=False))

print("\n---- example outputs ----")
for i in range(4):
    print("SRC:", " ".join(test_items[i]["src_tok"])); print("REF:", refs[i])
    for name in ["greedy", "beam k=5", "sample T=1.0", "top-p=0.9 (nucleus)"]: print(f"  {name:22s}: {dec_out[name][i]}")
    print()

### 4d. Attention heat-map (what is the decoder looking at?)

In [ ]:
def plot_attention(model, item):
    ids, att = translate(model, item["src_ids"], "greedy", want_attn=True)
    out_toks = tgt_vocab.decode(ids) + ["<eos>"]
    att = att[:len(out_toks)]
    plt.figure(figsize=(1 + 0.6 * len(item["src_tok"]), 1 + 0.45 * len(out_toks)))
    plt.imshow(att, cmap="viridis", aspect="auto")
    plt.xticks(range(len(item["src_tok"])), item["src_tok"], rotation=90)
    plt.yticks(range(len(out_toks)), out_toks)
    plt.xlabel("source"); plt.ylabel("generated"); plt.title(f"{model.kind} attention"); plt.colorbar(); plt.show()

# pick a mid-length sentence; if Devanagari glyphs render as boxes, install a Hindi font (e.g. Noto Sans Devanagari)
cand = [it for it in test_items if 8 <= len(it["src_tok"]) <= 14][:3]
for it in cand: plot_attention(models[best_kind], it)

In [ ]:
os.makedirs("results", exist_ok=True)
bleu_table.to_csv("results/exp2_attention_vs_none.csv", index=False)
dec_df.to_csv("results/exp2_decoding.csv", index=False)
len_df.to_csv("results/exp2_bleu_by_length.csv", index=False)

## 5. Notes for the write-up
* **Table 1:** BLEU no-attention vs. Bahdanau vs. Luong (section 4a) + BLEU-by-length plot (4b).
* **Table 2:** decoding strategies (4c) with example outputs. Typical findings: beam search gives a modest BLEU gain over greedy, with diminishing/negative returns for very large beams (short, generic outputs); sampling lowers BLEU but gives more varied output.
* Include one attention heat-map (4d) and comment on whether alignments look monotone/diagonal and whether word-order differences between the language pair show up.
* Stretch goal: a from-scratch Transformer (self-attention + positional encodings) trained on the same data for comparison.